### Trabalho Fase 2 do Curso de Pós-Graduação FIAP IA para Devs
#### Parte 7 - Tuning com Algoritmos Genéticos

Fonte de dados escolhida: DATASUS/SISCAN  
Tipo de dados de origem nesta etapa: Parquet

Arquivos utilizados:

```text
bases/treino/x_encoded.parquet
bases/treino/y.parquet
bases/teste/x_encoded.parquet
bases/teste/y.parquet
```

---

## Objetivo da Parte 7

O objetivo desta etapa é otimizar os hiperparâmetros do modelo SVM linear (`LinearSVC`) usando um Algoritmo Genético implementado diretamente no notebook. A otimização usa validação cruzada estratificada e uma função de fitness voltada ao problema de desbalanceamento da target.

O cromossomo de cada indivíduo representa uma configuração do SVM:

- `C`;
- `class_weight`;
- `scaler_type`;
- `max_iter`.

Breve descrição dos hiperparâmetros otimizados:

- `C`: controla a intensidade da regularização do `LinearSVC`. Valores menores aumentam a regularização e tendem a gerar uma fronteira de decisão mais simples; valores maiores reduzem a regularização e permitem maior ajuste aos dados de treino. O objetivo do AG é encontrar um equilíbrio entre generalização e capacidade de separar a classe positiva.
- `class_weight`: define pesos diferenciados para as classes. A opção `balanced` aumenta o peso da classe minoritária, ajudando o modelo a considerar melhor os casos positivos em uma base desbalanceada. O objetivo é melhorar recall e F1 da classe positiva.
- `scaler_type`: define se as features serão usadas sem escala, com padronização ou com normalização. Como SVM é sensível à escala das variáveis, esse gene testa qual transformação favorece a separação entre classes.
- `max_iter`: define o número máximo de iterações do otimizador interno. Valores maiores reduzem o risco de parada antes da convergência, mas aumentam o custo computacional. O objetivo é permitir convergência suficiente sem desperdiçar tempo de execução.

A coluna de idade mantida neste notebook é `CO_IDADE_PACIENTE_CAP`, seguindo o cenário consolidado para esta etapa.

---

## Índice / Sumário da Parte 7

**Item 1 - Leitura e preparação dos dados para SVM**

- Leitura das bases encoded geradas na Parte 2.
- Definição de constantes, espaços de busca e métricas usadas na avaliação.
- Seleção das colunas usadas pelo SVM.

**Item 2 - Amostragem estratificada**

- Criação de amostras estratificadas de treino e teste.
- Preservação aproximada da proporção da classe positiva.

**Item 3 - Execução do SVM com validação cruzada**

- Aplicação opcional de escala.
- Treinamento do `LinearSVC` em 5 folds estratificados.
- Retorno das métricas médias dos folds.

**Item 4 - Função de fitness**

- Cálculo da fitness com a combinação `0.7 * F1 + 0.3 * recall`.

**Item 5 - Representação genética e população inicial**

- Definição do cromossomo, solução base, variação inicial e criação dos indivíduos.

**Item 6 - Operadores genéticos**

- Avaliação da população.
- Seleção por torneio.
- Crossover uniforme.
- Mutação por gene.
- Elitismo e geração da nova população.

**Item 7 - Experimentos com Algoritmo Genético**

- Execução de três configurações de população, mutação e número de gerações.
- Registro do melhor indivíduo por geração.

**Item 8 - Avaliação final do SVM otimizado**

- Escolha do melhor indivíduo entre os três experimentos.
- Treinamento final sem validação cruzada.
- Avaliação na amostra estratificada de teste.

**Item 9 - Comparação com a Parte 4 e conclusão**

- Comparação entre o baseline SVM da Parte 4 e o SVM otimizado por Algoritmo Genético.
- Síntese dos ganhos, perdas e trade-offs observados nas métricas da classe positiva.


#### Item 1 - Leitura e preparação dos dados para SVM

A Parte 7 usa diretamente `x_encoded.parquet`, criado na Parte 2. Como a validação das bases já foi realizada na etapa de preparação, este bloco concentra os imports, a configuração das métricas, a leitura das bases e a definição dos espaços de busca do Algoritmo Genético.

Também é definida a estrutura `ClassificationMetrics`, usada para padronizar as métricas calculadas ao longo do notebook.


In [1]:
from pathlib import Path

import random
import time
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.svm import LinearSVC
from sklearn.preprocessing import StandardScaler, Normalizer
try:
    from IPython.display import display, HTML
except ImportError:
    def display(value):
        print(value)

    class HTML(str):
        pass
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)

display(HTML("""
<style>
.container {
    width: 100% !important;
    max-width: 100% !important;
}
.jp-Cell-outputArea,
.jp-OutputArea-output,
.output_area {
    width: 100% !important;
    max-width: 100% !important;
}
.dataframe {
    width: 100% !important;
}
.dataframe td,
.dataframe th {
    white-space: normal !important;
    word-break: break-word !important;
}
</style>
"""))

from dataclasses import dataclass

from sklearn.metrics import (
    accuracy_score,
    recall_score,
    f1_score,
    precision_score,
    roc_auc_score
)

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 120)

@dataclass
class ClassificationMetrics:
    accuracy: float
    recall: float
    f1: float
    precision: float
    roc_auc: float
    balanced_accuracy: float

def calculate_metrics(y_true, y_pred) -> ClassificationMetrics:
    return ClassificationMetrics(
        accuracy=accuracy_score(y_true, y_pred),
        recall=recall_score(y_true, y_pred, zero_division=0),
        f1=f1_score(y_true, y_pred, zero_division=0),
        precision=precision_score(y_true, y_pred, zero_division=0),
        roc_auc=roc_auc_score(y_true, y_pred),
        balanced_accuracy=balanced_accuracy_score(y_true, y_pred)
    )  


C_SPACE = [0.01, 0.1, 1, 10, 100]
CLASS_WEIGHT_SPACE = [None, "balanced"]
SCALER_SPACE = [None, "standard", "normalizer"]
MAX_ITER_SPACE = [1000, 3000, 5000, 10000]

RANDOM_STATE = 42
TARGET_COLUMN = 'TARGET_CANCER_MAMA_PROVAVEL'
DATA_DIR = Path('bases')

N_TRAIN_SVM = 12000
N_VALIDATION_SVM = 4000
N_TEST_SVM = 8000

input_files = {
    'x_train_encoded': DATA_DIR / 'treino' / 'x_encoded.parquet',
    'y_train': DATA_DIR / 'treino' / 'y.parquet',
    'x_test_encoded': DATA_DIR / 'teste' / 'x_encoded.parquet',
    'y_test': DATA_DIR / 'teste' / 'y.parquet',
}

X_train_encoded = pd.read_parquet(input_files['x_train_encoded'])
y_train = pd.read_parquet(input_files['y_train'])[TARGET_COLUMN]
X_test_encoded = pd.read_parquet(input_files['x_test_encoded'])
y_test = pd.read_parquet(input_files['y_test'])[TARGET_COLUMN]

X_train_encoded = X_train_encoded.drop(columns=['CO_IDADE_PACIENTE_NUM'])
X_test_encoded = X_test_encoded.drop(columns=['CO_IDADE_PACIENTE_NUM'])

age_columns = ['CO_IDADE_PACIENTE_CAP']

context_numeric_columns = [
    column for column in ['CO_TEMPO_MAMO_ANTERIOR_NUM']
    if column in X_train_encoded.columns
]
encoded_categorical_columns = [
    column for column in X_train_encoded.columns
    if column not in age_columns + context_numeric_columns
]

svm_input_summary = {
    'x_train_encoded_shape': X_train_encoded.shape,
    'x_test_encoded_shape': X_test_encoded.shape,
    'compared_age_columns': age_columns,
    'context_numeric_column_count': len(context_numeric_columns),
    'context_numeric_columns': context_numeric_columns,
    'encoded_categorical_column_count': len(encoded_categorical_columns),
}
svm_input_summary


{'x_train_encoded_shape': (213978, 24),
 'x_test_encoded_shape': (53495, 24),
 'compared_age_columns': ['CO_IDADE_PACIENTE_CAP'],
 'context_numeric_column_count': 1,
 'context_numeric_columns': ['CO_TEMPO_MAMO_ANTERIOR_NUM'],
 'encoded_categorical_column_count': 22}

#### Item 2 - Amostragem estratificada

A amostragem estratificada preserva aproximadamente a proporção da target, que é fortemente desbalanceada. A amostra de treino é usada pelo Algoritmo Genético com validação cruzada; a amostra de teste fica reservada para a avaliação final do melhor indivíduo. Para manter a comparação com a Parte 4, a amostra final de teste usa `N_TEST_SVM = 8000` registros.


In [2]:
def limit_stratified_sample(X, y, sample_size, random_state=RANDOM_STATE):
    if sample_size is None or sample_size >= len(X):
        return X.copy(), y.copy()

    _, X_sample, _, y_sample = train_test_split(
        X,
        y,
        test_size=sample_size,
        random_state=random_state,
        stratify=y,
    )
    return X_sample.reset_index(drop=True), y_sample.reset_index(drop=True)


X_train_sample, y_train_sample = limit_stratified_sample(
    X_train_encoded,
    y_train,
    N_TRAIN_SVM + N_VALIDATION_SVM,
)

X_test_sample, y_test_sample = limit_stratified_sample(
    X_test_encoded,
    y_test,
    N_TEST_SVM,
)

svm_samples_summary = pd.DataFrame([
    {'base': 'treino_svm', 'linhas': len(X_train_sample), 'classe_positiva': int(y_train_sample.sum()), 'percentual_positivo': round(y_train_sample.mean() * 100, 2)},
    {'base': 'teste_svm', 'linhas': len(X_test_sample), 'classe_positiva': int(y_test_sample.sum()), 'percentual_positivo': round(y_test_sample.mean() * 100, 2)},
])
svm_samples_summary


,base,linhas,classe_positiva,percentual_positivo
0,treino_svm,16000,679,4.24
1,teste_svm,8000,339,4.24


#### Item 3 - Execução do SVM com validação cruzada

O bloco abaixo reúne a aplicação opcional de escala e a avaliação do `LinearSVC`. Para cada indivíduo, o modelo é treinado em 5 folds estratificados, e as métricas finais retornadas são as médias dos folds.

Essa função é usada durante o Algoritmo Genético para avaliar cada combinação de hiperparâmetros sem usar o conjunto de teste.


In [3]:


def apply_scaler(x_train, x_test, scaler_type):
    if scaler_type == "standard":
        scaler = StandardScaler()
        X_train_processed = scaler.fit_transform(x_train)
        X_test_processed = scaler.transform(x_test)
        return X_train_processed, X_test_processed

    if scaler_type == "normalizer":
        scaler = Normalizer()
        X_train_processed = scaler.fit_transform(x_train)
        X_test_processed = scaler.transform(x_test)
        return X_train_processed, X_test_processed

    if scaler_type is None:
        return x_train, x_test

    raise ValueError("scaler_type deve ser 'standard', 'normalizer' ou None")


def execute_linear_svm(
    x,
    y,
    C=1.0,
    class_weight=None,
    loss="squared_hinge",
    penalty="l2",
    dual="auto",
    scaler_type="standard",
    max_iter=5000,
    show=False,
    short_show=False
):    
    n_splits = 5
    skf = StratifiedKFold(n_splits, shuffle=True, random_state=42)
    
    metrics = ClassificationMetrics(
            accuracy=0.0,
            recall=0.0,
            f1=0.0,
            precision=0.0,
            roc_auc=0.0,
            balanced_accuracy=0.0
        )

    for train_idx, val_idx in skf.split(x, y):
         X_tr, X_val = x.iloc[train_idx], x.iloc[val_idx]
         y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

         X_train_processed, X_test_processed = apply_scaler(X_tr, X_val, scaler_type) 

         classifier = LinearSVC(
            C=C,
            class_weight=class_weight,
            loss=loss,
            penalty=penalty,
            dual=dual,
            max_iter=max_iter,
            random_state=42
         )
 
         classifier.fit(X_train_processed, y_tr)
         y_pred = classifier.predict(X_test_processed)

         current_execution = calculate_metrics(y_val, y_pred)    
         
         metrics.accuracy += current_execution.accuracy
         metrics.recall += current_execution.recall
         metrics.f1 += current_execution.f1
         metrics.precision += current_execution.precision
         metrics.roc_auc += current_execution.roc_auc
         metrics.balanced_accuracy += current_execution.balanced_accuracy

    metrics.accuracy /= n_splits
    metrics.recall /= n_splits
    metrics.f1 /= n_splits
    metrics.precision /= n_splits
    metrics.roc_auc /= n_splits
    metrics.balanced_accuracy /= n_splits

    if show:
        if short_show:
            print(f'recall: {metrics.recall} f1: {metrics.f1}')
        else:       
            print(f'########### LinearSVC ##############')
            print(f'C: {C}')
            print(f'class_weight: {class_weight}')
            print(f'loss: {loss}')
            print(f'penalty: {penalty}')
            print(f'dual: {dual}')
            print(f'scaler_type: {scaler_type}')
            print(f'max_iter: {max_iter}')
            print(f'accuracy: {metrics.accuracy}')
            print(f'recall: {metrics.recall}')
            print(f'f1: {metrics.f1}')
            print(f'precision: {metrics.precision}')
            print(f'roc_auc: {metrics.roc_auc}')
            print(f'balanced_accuracy: {metrics.balanced_accuracy}')
            print('######################################')     

    return metrics

#### Item 4 - Função de fitness

A fitness prioriza a classe positiva, que é minoritária. A métrica principal combina F1 e recall:

```text
fitness = 0.7 * F1 + 0.3 * recall
```

O F1 equilibra precision e recall, enquanto o recall aumenta a importância de recuperar casos positivos.


In [4]:
def fitness(individual, show=False, short_show=False) -> float:
    svm = execute_linear_svm(
        x = X_train_sample,
        y = y_train_sample,
        C = individual["C"],
        class_weight = individual["class_weight"],
        scaler_type = individual["scaler_type"],
        max_iter = individual["max_iter"],
        show = show,
        short_show = short_show
    )

    return calculate_svm_execution_fitness(svm)

def calculate_svm_execution_fitness(svm_execution):
    return 0.7 * svm_execution.f1 + 0.3 * svm_execution.recall


#### Item 5 - Representação genética e população inicial

Cada indivíduo representa um conjunto de hiperparâmetros do SVM. A população inicial é criada a partir de uma solução base com variações aleatórias, estratégia usada como hot start para iniciar a busca em uma região plausível do espaço de hiperparâmetros.


In [5]:
def create_random_individuals():
    return [{
        "C": random.choice(C_SPACE),
        "class_weight": random.choice(CLASS_WEIGHT_SPACE),
        "scaler_type": random.choice(SCALER_SPACE),
        "max_iter": random.choice(MAX_ITER_SPACE)
    } for _ in range(INDIVIDUAL_COUNT)]

BASE_SVM_SOLUTION = {
    "C": 0.1,
    "class_weight": "balanced",
    "scaler_type": "standard",
    "max_iter": 5000
}

VARIATION_RATE = 0.3

def create_svm_variation(base_solution, variation_rate):
    individual = base_solution.copy()

    if random.random() < variation_rate:
        individual["C"] = random.choice(C_SPACE)

    if random.random() < variation_rate:
        individual["class_weight"] = random.choice(CLASS_WEIGHT_SPACE)

    if random.random() < variation_rate:
        individual["scaler_type"] = random.choice(SCALER_SPACE)

    if random.random() < variation_rate:
        individual["max_iter"] = random.choice(MAX_ITER_SPACE)

    return individual

def create_hot_start_individuals(individual_count):
    return [create_svm_variation(BASE_SVM_SOLUTION, VARIATION_RATE) for _ in range(individual_count) ] 

#### Item 6.1 - Avaliação da população

Cada indivíduo recebe uma nota de fitness calculada pela validação cruzada. Em seguida, a população é ordenada da maior para a menor fitness, permitindo selecionar os melhores candidatos para reprodução e elitismo.


In [6]:
def evaluate_population(individuals):   
    for individual in individuals:
        individual['evaluation'] = fitness(individual)

    sorted_population = sorted(
        individuals,
        key=lambda item: item["evaluation"],
        reverse=True
    )
    
    return sorted_population           


#### Item 6.2 - Seleção por torneio

A seleção por torneio escolhe alguns candidatos aleatoriamente e retorna o indivíduo com maior fitness entre eles. Esse operador mantém pressão seletiva sem depender de probabilidades proporcionais à fitness.


In [7]:
def tournament_selection(population, k=3): 
    candidates = random.sample(population, k)
    return max(candidates, key=lambda x: x["evaluation"])

#### Item 6.3 - Crossover uniforme

O crossover uniforme cria um novo indivíduo misturando genes de dois pais. Para cada hiperparâmetro, o valor é herdado de um dos pais com probabilidade igual.


In [8]:
def uniform_crossover(parent1, parent2, crossover_rate=0.85):
    if random.random() > crossover_rate:
        selected = random.choice([parent1, parent2])
        child = selected.copy()
        return child

    child = {}

    for key in parent1.keys():
        if random.random() < 0.5:
            child[key] = parent1[key]
        else:
            child[key] = parent2[key]

    return child

#### Item 6.4 - Mutação por gene

A mutação altera cada gene com uma probabilidade definida pelo experimento. Esse operador mantém diversidade na população e ajuda a explorar configurações que não apareceriam apenas por crossover.


In [9]:
def svm_mutation(individual, mutation_rate=0.20):
    new = individual.copy()

    if random.random() < mutation_rate:
        new["max_iter"] = random.choice(MAX_ITER_SPACE)

    if random.random() < mutation_rate:
        new["C"] = random.choice(C_SPACE)   

    if random.random() < mutation_rate:
        new["class_weight"] = random.choice(CLASS_WEIGHT_SPACE)

    if random.random() < mutation_rate:
        new["scaler_type"] = random.choice(SCALER_SPACE)

    return new

#### Item 6.5 - Nova geração com elitismo

A nova geração preserva os melhores indivíduos da geração atual e completa o restante da população com filhos gerados por seleção, crossover e mutação. O elitismo evita perder a melhor solução já encontrada.


In [10]:
def generate_new_generation(
    sorted_by_fitness_current_population,
    population_size,
    crossover_rate=0.85,
    mutation_rate=0.20,
    elite_size=2
):
    new_generation = []    

    
    elite = sorted_by_fitness_current_population[:elite_size]
    new_generation.extend([ind.copy() for ind in elite])

    while len(new_generation) < population_size:
        parent1 = tournament_selection(sorted_by_fitness_current_population)
        parent2 = tournament_selection(sorted_by_fitness_current_population)

        child = uniform_crossover(
            parent1,
            parent2,
            crossover_rate=crossover_rate
        )

        child = svm_mutation(
            child,
            mutation_rate=mutation_rate
        )   
        new_generation.append(child)

    return new_generation

#### Item 6.6 - Execução do Algoritmo Genético com monitoramento

A função principal executa as gerações do Algoritmo Genético. Em cada geração, a população é avaliada, o melhor indivíduo é registrado e o histórico armazena informações para tracking de desempenho: melhor fitness, fitness média, desvio padrão, pior fitness, tempo da geração e configuração do experimento.

Esses registros funcionam como logging estruturado do treinamento e permitem comparar custo computacional e evolução da população entre diferentes configurações do AG.


In [11]:
def extract_fitness_values(population):
    return [individual["evaluation"] for individual in population]


def execute_ga(
    generation_count,
    population_size,
    crossover_rate,
    mutation_rate,
    elite_size,
    experiment_name="Experimento",
):
    current_generation = None
    history_data = []
    best_individuals = []
    experiment_started_at = time.perf_counter()

    for i in range(0, generation_count):
        generation_started_at = time.perf_counter()
        current_generation = generate_new_generation(
            sorted_by_fitness_current_population,
            population_size=population_size,
            crossover_rate=crossover_rate,
            mutation_rate=mutation_rate,
            elite_size=elite_size,
        ) if current_generation is not None else create_hot_start_individuals(population_size)

        sorted_by_fitness_current_population = evaluate_population(current_generation)
        best_evaluation = sorted_by_fitness_current_population[0]
        fitness_values = extract_fitness_values(sorted_by_fitness_current_population)
        generation_elapsed_seconds = time.perf_counter() - generation_started_at

        history_data.append({
            "Experimento": experiment_name,
            "Geração": i + 1,
            "População": population_size,
            "Taxa crossover": crossover_rate,
            "Taxa mutação": mutation_rate,
            "Elitismo": elite_size,
            "Melhor fitness": round(float(max(fitness_values)), 6),
            "Fitness média": round(float(np.mean(fitness_values)), 6),
            "Fitness desvio": round(float(np.std(fitness_values)), 6),
            "Pior fitness": round(float(min(fitness_values)), 6),
            "Tempo geração (s)": round(generation_elapsed_seconds, 3),
            "Melhor Indivíduo": best_evaluation.copy(),
        })
        best_individuals.append(best_evaluation.copy())

    top_individual = sorted(
        best_individuals,
        key=lambda item: item["evaluation"],
        reverse=True,
    )[0]

    experiment_elapsed_seconds = time.perf_counter() - experiment_started_at
    experiment_log = {
        "Experimento": experiment_name,
        "Gerações": generation_count,
        "População": population_size,
        "Taxa crossover": crossover_rate,
        "Taxa mutação": mutation_rate,
        "Elitismo": elite_size,
        "Melhor fitness": round(float(top_individual["evaluation"]), 6),
        "Tempo total (s)": round(experiment_elapsed_seconds, 3),
        "Tempo médio por geração (s)": round(experiment_elapsed_seconds / generation_count, 3),
        "Melhor Indivíduo": top_individual.copy(),
    }

    return pd.DataFrame(history_data), top_individual, experiment_log


#### Item 7 - Execução dos três experimentos e logs de desempenho

O trabalho exige pelo menos três experimentos com configurações diferentes do Algoritmo Genético. As execuções abaixo variam tamanho da população, taxa de mutação e número de gerações, mantendo o mesmo crossover e elitismo.

Além do melhor indivíduo, cada experimento gera logs estruturados para tracking de desempenho: fitness por geração, estatísticas da população e tempo de execução.


In [12]:
experiment_1_history, experiment_1_best, experiment_1_log = execute_ga(
    experiment_name="Experimento 1",
    generation_count=20,
    population_size=20,
    crossover_rate=0.85,
    mutation_rate=0.05,
    elite_size=2,
)

experiment_2_history, experiment_2_best, experiment_2_log = execute_ga(
    experiment_name="Experimento 2",
    generation_count=20,
    population_size=40,
    crossover_rate=0.85,
    mutation_rate=0.10,
    elite_size=2,
)

experiment_3_history, experiment_3_best, experiment_3_log = execute_ga(
    experiment_name="Experimento 3",
    generation_count=30,
    population_size=40,
    crossover_rate=0.85,
    mutation_rate=0.20,
    elite_size=2,
)

ga_experiment_summary = pd.DataFrame([
    experiment_1_log,
    experiment_2_log,
    experiment_3_log,
])

display(ga_experiment_summary)


,Experimento,Gerações,População,Taxa crossover,Taxa mutação,Elitismo,Melhor fitness,Tempo total (s),Tempo médio por geração (s),Melhor Indivíduo
0,Experimento 1,20,20,0.85,0.05,2,0.276383,39.187,1.959,"{'C': 0.01, 'class_weight': 'balanced', 'scaler_type': 'normalizer', 'max_iter': 5000, 'evaluation': 0.2763833043915179}"
1,Experimento 2,20,40,0.85,0.10,2,0.276383,91.726,4.586,"{'C': 0.01, 'class_weight': 'balanced', 'scaler_type': 'normalizer', 'max_iter': 5000, 'evaluation': 0.2763833043915179}"
2,Experimento 3,30,40,0.85,0.20,2,0.276383,137.692,4.590,"{'C': 0.01, 'class_weight': 'balanced', 'scaler_type': 'normalizer', 'max_iter': 5000, 'evaluation': 0.2763833043915179}"


#### Item 7.1 - Histórico e tracking do Experimento 1

Este histórico mostra o melhor indivíduo encontrado em cada geração do primeiro experimento, junto com métricas de monitoramento da população e tempo por geração.


In [13]:
display(experiment_1_history)

,Experimento,Geração,População,Taxa crossover,Taxa mutação,Elitismo,Melhor fitness,Fitness média,Fitness desvio,Pior fitness,Tempo geração (s),Melhor Indivíduo
0,Experimento 1,1,20,0.85,0.05,2,0.276383,0.216708,0.073445,0.000000,6.685,"{'C': 0.01, 'class_weight': 'balanced', 'scaler_type': 'normalizer', 'max_iter': 5000, 'evaluation': 0.2763833043915179}"
1,Experimento 1,2,20,0.85,0.05,2,0.276383,0.254756,0.016245,0.233462,4.022,"{'C': 0.01, 'class_weight': 'balanced', 'scaler_type': 'normalizer', 'max_iter': 5000, 'evaluation': 0.2763833043915179}"
2,Experimento 1,3,20,0.85,0.05,2,0.276383,0.265205,0.011878,0.233590,1.795,"{'C': 0.01, 'class_weight': 'balanced', 'scaler_type': 'normalizer', 'max_iter': 5000, 'evaluation': 0.2763833043915179}"
3,Experimento 1,4,20,0.85,0.05,2,0.276383,0.258932,0.059962,0.000000,1.594,"{'C': 0.01, 'class_weight': 'balanced', 'scaler_type': 'normalizer', 'max_iter': 5000, 'evaluation': 0.2763833043915179}"
4,Experimento 1,5,20,0.85,0.05,2,0.276383,0.271443,0.012941,0.233590,1.891,"{'C': 0.01, 'class_weight': 'balanced', 'scaler_type': 'normalizer', 'max_iter': 5000, 'evaluation': 0.2763833043915179}"
5,Experimento 1,6,20,0.85,0.05,2,0.276383,0.276383,0.000000,0.276383,1.490,"{'C': 0.01, 'class_weight': 'balanced', 'scaler_type': 'normalizer', 'max_iter': 5000, 'evaluation': 0.2763833043915179}"
6,Experimento 1,7,20,0.85,0.05,2,0.276383,0.274944,0.006273,0.247600,1.513,"{'C': 0.01, 'class_weight': 'balanced', 'scaler_type': 'normalizer', 'max_iter': 5000, 'evaluation': 0.2763833043915179}"
7,Experimento 1,8,20,0.85,0.05,2,0.276383,0.246633,0.082722,0.000000,1.507,"{'C': 0.01, 'class_weight': 'balanced', 'scaler_type': 'normalizer', 'max_iter': 5000, 'evaluation': 0.2763833043915179}"
8,Experimento 1,9,20,0.85,0.05,2,0.276383,0.270720,0.013703,0.234144,1.562,"{'C': 0.01, 'class_weight': 'balanced', 'scaler_type': 'normalizer', 'max_iter': 5000, 'evaluation': 0.2763833043915179}"
9,Experimento 1,10,20,0.85,0.05,2,0.276383,0.261125,0.060233,0.000000,1.495,"{'C': 0.01, 'class_weight': 'balanced', 'scaler_type': 'normalizer', 'max_iter': 5000, 'evaluation': 0.2763833043915179}"


#### Item 7.2 - Histórico e tracking do Experimento 2

Este histórico mostra o melhor indivíduo encontrado em cada geração do segundo experimento, junto com métricas de monitoramento da população e tempo por geração.


In [14]:
display(experiment_2_history)

,Experimento,Geração,População,Taxa crossover,Taxa mutação,Elitismo,Melhor fitness,Fitness média,Fitness desvio,Pior fitness,Tempo geração (s),Melhor Indivíduo
0,Experimento 2,1,40,0.85,0.1,2,0.263160,0.153972,0.113244,0.000000,11.713,"{'C': 0.1, 'class_weight': 'balanced', 'scaler_type': 'normalizer', 'max_iter': 10000, 'evaluation': 0.263160067847973}"
1,Experimento 2,2,40,0.85,0.1,2,0.276383,0.229902,0.054425,0.000000,10.513,"{'C': 0.01, 'class_weight': 'balanced', 'scaler_type': 'normalizer', 'max_iter': 5000, 'evaluation': 0.2763833043915179}"
2,Experimento 2,3,40,0.85,0.1,2,0.276383,0.214573,0.091449,0.000000,6.434,"{'C': 0.01, 'class_weight': 'balanced', 'scaler_type': 'normalizer', 'max_iter': 5000, 'evaluation': 0.2763833043915179}"
3,Experimento 2,4,40,0.85,0.1,2,0.276383,0.246372,0.058791,0.000000,5.283,"{'C': 0.01, 'class_weight': 'balanced', 'scaler_type': 'normalizer', 'max_iter': 5000, 'evaluation': 0.2763833043915179}"
4,Experimento 2,5,40,0.85,0.1,2,0.276383,0.261794,0.043257,0.000000,3.319,"{'C': 0.01, 'class_weight': 'balanced', 'scaler_type': 'normalizer', 'max_iter': 5000, 'evaluation': 0.2763833043915179}"
5,Experimento 2,6,40,0.85,0.1,2,0.276383,0.254289,0.060404,0.000000,3.943,"{'C': 0.01, 'class_weight': 'balanced', 'scaler_type': 'normalizer', 'max_iter': 5000, 'evaluation': 0.2763833043915179}"
6,Experimento 2,7,40,0.85,0.1,2,0.276383,0.240229,0.091069,0.000000,3.049,"{'C': 0.01, 'class_weight': 'balanced', 'scaler_type': 'normalizer', 'max_iter': 5000, 'evaluation': 0.2763833043915179}"
7,Experimento 2,8,40,0.85,0.1,2,0.276383,0.262040,0.044941,0.000000,3.623,"{'C': 0.01, 'class_weight': 'balanced', 'scaler_type': 'normalizer', 'max_iter': 5000, 'evaluation': 0.2763833043915179}"
8,Experimento 2,9,40,0.85,0.1,2,0.276383,0.250915,0.072557,0.000000,3.692,"{'C': 0.01, 'class_weight': 'balanced', 'scaler_type': 'normalizer', 'max_iter': 5000, 'evaluation': 0.2763833043915179}"
9,Experimento 2,10,40,0.85,0.1,2,0.276383,0.265189,0.044010,0.000000,4.648,"{'C': 0.01, 'class_weight': 'balanced', 'scaler_type': 'normalizer', 'max_iter': 5000, 'evaluation': 0.2763833043915179}"


#### Item 7.3 - Histórico e tracking do Experimento 3

Este histórico mostra o melhor indivíduo encontrado em cada geração do terceiro experimento, junto com métricas de monitoramento da população e tempo por geração.


In [15]:
display(experiment_3_history)

,Experimento,Geração,População,Taxa crossover,Taxa mutação,Elitismo,Melhor fitness,Fitness média,Fitness desvio,Pior fitness,Tempo geração (s),Melhor Indivíduo
0,Experimento 3,1,40,0.85,0.2,2,0.263160,0.185923,0.100709,0.0,11.733,"{'C': 0.1, 'class_weight': 'balanced', 'scaler_type': 'normalizer', 'max_iter': 5000, 'evaluation': 0.263160067847973}"
1,Experimento 3,2,40,0.85,0.2,2,0.263160,0.215077,0.082314,0.0,9.591,"{'C': 0.1, 'class_weight': 'balanced', 'scaler_type': 'normalizer', 'max_iter': 5000, 'evaluation': 0.263160067847973}"
2,Experimento 3,3,40,0.85,0.2,2,0.276383,0.234317,0.068397,0.0,6.381,"{'C': 0.01, 'class_weight': 'balanced', 'scaler_type': 'normalizer', 'max_iter': 5000, 'evaluation': 0.2763833043915179}"
3,Experimento 3,4,40,0.85,0.2,2,0.276383,0.235225,0.079891,0.0,5.132,"{'C': 0.01, 'class_weight': 'balanced', 'scaler_type': 'normalizer', 'max_iter': 5000, 'evaluation': 0.2763833043915179}"
4,Experimento 3,5,40,0.85,0.2,2,0.276383,0.241451,0.070825,0.0,4.614,"{'C': 0.01, 'class_weight': 'balanced', 'scaler_type': 'normalizer', 'max_iter': 5000, 'evaluation': 0.2763833043915179}"
5,Experimento 3,6,40,0.85,0.2,2,0.276383,0.248452,0.072120,0.0,3.434,"{'C': 0.01, 'class_weight': 'balanced', 'scaler_type': 'normalizer', 'max_iter': 5000, 'evaluation': 0.2763833043915179}"
6,Experimento 3,7,40,0.85,0.2,2,0.276383,0.254989,0.060446,0.0,3.621,"{'C': 0.01, 'class_weight': 'balanced', 'scaler_type': 'normalizer', 'max_iter': 5000, 'evaluation': 0.2763833043915179}"
7,Experimento 3,8,40,0.85,0.2,2,0.276383,0.255023,0.060552,0.0,3.488,"{'C': 0.01, 'class_weight': 'balanced', 'scaler_type': 'normalizer', 'max_iter': 5000, 'evaluation': 0.2763833043915179}"
8,Experimento 3,9,40,0.85,0.2,2,0.276383,0.259577,0.044790,0.0,4.036,"{'C': 0.01, 'class_weight': 'balanced', 'scaler_type': 'normalizer', 'max_iter': 5000, 'evaluation': 0.2763833043915179}"
9,Experimento 3,10,40,0.85,0.2,2,0.276383,0.242517,0.081978,0.0,3.486,"{'C': 0.01, 'class_weight': 'balanced', 'scaler_type': 'normalizer', 'max_iter': 5000, 'evaluation': 0.2763833043915179}"


#### Item 7.4 - Validação cruzada do melhor indivíduo do Experimento 1

A célula abaixo reexibe as métricas médias de validação cruzada para o melhor indivíduo encontrado no primeiro experimento.


In [16]:
execute_linear_svm(
        x = X_train_sample,
        y = y_train_sample,
        C = experiment_1_best["C"],
        class_weight = experiment_1_best["class_weight"],
        scaler_type = experiment_1_best["scaler_type"],
        max_iter = experiment_1_best["max_iter"],
        show = True,
        short_show=False
    )

########### LinearSVC ##############
C: 0.01
class_weight: balanced
loss: squared_hinge
penalty: l2
dual: auto
scaler_type: normalizer
max_iter: 5000
accuracy: 0.45024999999999993
recall: 0.6951851851851851
f1: 0.0968967840513749
precision: 0.052078524327829
roc_auc: 0.567291277313117
balanced_accuracy: 0.5672912773131171
######################################


ClassificationMetrics(accuracy=0.45024999999999993, recall=0.6951851851851851, f1=0.0968967840513749, precision=0.052078524327829, roc_auc=0.567291277313117, balanced_accuracy=0.5672912773131171)

#### Item 7.5 - Validação cruzada do melhor indivíduo do Experimento 2

A célula abaixo reexibe as métricas médias de validação cruzada para o melhor indivíduo encontrado no segundo experimento.


In [17]:
execute_linear_svm(
        x = X_train_sample,
        y = y_train_sample,
        C = experiment_2_best["C"],
        class_weight = experiment_2_best["class_weight"],
        scaler_type = experiment_2_best["scaler_type"],
        max_iter = experiment_2_best["max_iter"],
        show = True,
        short_show=False
    )

########### LinearSVC ##############
C: 0.01
class_weight: balanced
loss: squared_hinge
penalty: l2
dual: auto
scaler_type: normalizer
max_iter: 5000
accuracy: 0.45024999999999993
recall: 0.6951851851851851
f1: 0.0968967840513749
precision: 0.052078524327829
roc_auc: 0.567291277313117
balanced_accuracy: 0.5672912773131171
######################################


ClassificationMetrics(accuracy=0.45024999999999993, recall=0.6951851851851851, f1=0.0968967840513749, precision=0.052078524327829, roc_auc=0.567291277313117, balanced_accuracy=0.5672912773131171)

#### Item 7.6 - Validação cruzada do melhor indivíduo do Experimento 3

A célula abaixo reexibe as métricas médias de validação cruzada para o melhor indivíduo encontrado no terceiro experimento.


In [18]:
execute_linear_svm(
        x = X_train_sample,
        y = y_train_sample,
        C = experiment_3_best["C"],
        class_weight = experiment_3_best["class_weight"],
        scaler_type = experiment_3_best["scaler_type"],
        max_iter = experiment_3_best["max_iter"],
        show = True,
        short_show=False
    )

########### LinearSVC ##############
C: 0.01
class_weight: balanced
loss: squared_hinge
penalty: l2
dual: auto
scaler_type: normalizer
max_iter: 5000
accuracy: 0.45024999999999993
recall: 0.6951851851851851
f1: 0.0968967840513749
precision: 0.052078524327829
roc_auc: 0.567291277313117
balanced_accuracy: 0.5672912773131171
######################################


ClassificationMetrics(accuracy=0.45024999999999993, recall=0.6951851851851851, f1=0.0968967840513749, precision=0.052078524327829, roc_auc=0.567291277313117, balanced_accuracy=0.5672912773131171)

#### Item 8 - Avaliação final do SVM otimizado

Depois dos três experimentos, o melhor indivíduo é escolhido pela maior fitness média. O modelo final é treinado uma única vez com a amostra de treino e avaliado na amostra estratificada de teste.

Nesta etapa não há validação cruzada: o teste é usado apenas para medir o desempenho final do modelo otimizado pelo Algoritmo Genético.


In [19]:
best_individual = max([experiment_1_best, experiment_2_best, experiment_3_best], key=lambda x: x["evaluation"])

final_model = LinearSVC(
            C=best_individual["C"],
            class_weight = best_individual["class_weight"],           
            max_iter = best_individual["max_iter"],
            loss="squared_hinge",
            penalty="l2",
            dual="auto",                   
            random_state=42
         )

X_train_final, X_test_final = apply_scaler(
      X_train_sample,
      X_test_sample,
      best_individual["scaler_type"]
  )
 
final_model.fit(X_train_final, y_train_sample)
y_pred = final_model.predict(X_test_final)
current_execution = calculate_metrics(y_test_sample, y_pred)  
display(current_execution)

ClassificationMetrics(accuracy=0.448, recall=0.6902654867256637, f1=0.09582309582309582, precision=0.05148514851485148, roc_auc=0.5637726076103191, balanced_accuracy=0.5637726076103191)

#### Item 9 - Comparação com o baseline SVM da Parte 4

A comparação abaixo usa como referência o SVM tunado com `GridSearchCV` na Parte 4. Como a base é desbalanceada, a leitura principal deve priorizar F1, recall, precision e balanced accuracy, e não apenas accuracy.

O modelo otimizado por Algoritmo Genético é avaliado com o melhor indivíduo encontrado nos três experimentos anteriores, usando uma amostra estratificada de teste com 8.000 registros, mesma escala usada no baseline da Parte 4.


In [20]:
baseline_part_4_metrics = {
    "modelo": "SVM baseline Parte 4 (GridSearchCV)",
    "accuracy": 0.6811,
    "balanced_accuracy": 0.6066,
    "precision": 0.0693,
    "recall": 0.5251,
    "f1": 0.1225,
    "roc_auc": 0.6492,
}

optimized_ga_metrics = {
    "modelo": "SVM otimizado com Algoritmo Genético",
    "accuracy": round(current_execution.accuracy, 4),
    "balanced_accuracy": round(current_execution.balanced_accuracy, 4),
    "precision": round(current_execution.precision, 4),
    "recall": round(current_execution.recall, 4),
    "f1": round(current_execution.f1, 4),
    "roc_auc": round(current_execution.roc_auc, 4),
}

metrics_comparison = pd.DataFrame([
    baseline_part_4_metrics,
    optimized_ga_metrics,
])

metrics_delta = {
    "modelo": "Diferença AG - baseline",
    "accuracy": round(optimized_ga_metrics["accuracy"] - baseline_part_4_metrics["accuracy"], 4),
    "balanced_accuracy": round(optimized_ga_metrics["balanced_accuracy"] - baseline_part_4_metrics["balanced_accuracy"], 4),
    "precision": round(optimized_ga_metrics["precision"] - baseline_part_4_metrics["precision"], 4),
    "recall": round(optimized_ga_metrics["recall"] - baseline_part_4_metrics["recall"], 4),
    "f1": round(optimized_ga_metrics["f1"] - baseline_part_4_metrics["f1"], 4),
    "roc_auc": round(optimized_ga_metrics["roc_auc"] - baseline_part_4_metrics["roc_auc"], 4),
}

metrics_comparison = pd.concat(
    [metrics_comparison, pd.DataFrame([metrics_delta])],
    ignore_index=True,
)

metrics_comparison


,modelo,accuracy,balanced_accuracy,precision,recall,f1,roc_auc
0,SVM baseline Parte 4 (GridSearchCV),0.6811,0.6066,0.0693,0.5251,0.1225,0.6492
1,SVM otimizado com Algoritmo Genético,0.4480,0.5638,0.0515,0.6903,0.0958,0.5638
2,Diferença AG - baseline,-0.2331,-0.0428,-0.0178,0.1652,-0.0267,-0.0854


#### Item 9.1 - Relatório final do SVM otimizado

O relatório de classificação e a matriz de confusão detalham o comportamento do modelo final na amostra de teste. Esses resultados ajudam a interpretar o trade-off entre recuperar mais casos positivos e manter precisão aceitável.


In [21]:
final_classification_report = pd.DataFrame(
    classification_report(
        y_test_sample,
        y_pred,
        output_dict=True,
        zero_division=0,
    )
).T

final_confusion_matrix = pd.DataFrame(
    confusion_matrix(y_test_sample, y_pred),
    index=["real_0", "real_1"],
    columns=["pred_0", "pred_1"],
)

display(final_classification_report)
display(final_confusion_matrix)


,precision,recall,f1-score,support
0,0.969609,0.437280,0.602735,7661.000
1,0.051485,0.690265,0.095823,339.000
accuracy,0.448000,0.448000,0.448000,0.448
macro avg,0.510547,0.563773,0.349279,8000.000
weighted avg,0.930704,0.448000,0.581254,8000.000


,pred_0,pred_1
real_0,3350,4311
real_1,105,234


#### Item 9.2 - Conclusão da otimização com Algoritmo Genético

A conclusão abaixo sintetiza a melhor configuração encontrada pelo Algoritmo Genético, compara o resultado com o baseline da Parte 4 e registra a interpretação das métricas mais importantes para a classe positiva.


In [22]:
f1_delta = metrics_delta["f1"]
recall_delta = metrics_delta["recall"]
precision_delta = metrics_delta["precision"]
balanced_accuracy_delta = metrics_delta["balanced_accuracy"]

conclusion_rows = [
    {
        "aspecto": "melhor_individuo_ag",
        "conclusao": f"C={best_individual['C']}, class_weight={best_individual['class_weight']}, scaler_type={best_individual['scaler_type']}, max_iter={best_individual['max_iter']}, fitness={best_individual['evaluation']:.4f}.",
    },
    {
        "aspecto": "comparacao_f1",
        "conclusao": f"O F1 do SVM com AG foi {optimized_ga_metrics['f1']:.4f}, contra {baseline_part_4_metrics['f1']:.4f} no baseline da Parte 4. Diferença: {f1_delta:+.4f}.",
    },
    {
        "aspecto": "comparacao_recall",
        "conclusao": f"O recall do SVM com AG foi {optimized_ga_metrics['recall']:.4f}, contra {baseline_part_4_metrics['recall']:.4f} no baseline. Diferença: {recall_delta:+.4f}.",
    },
    {
        "aspecto": "comparacao_precision",
        "conclusao": f"A precision do SVM com AG foi {optimized_ga_metrics['precision']:.4f}, contra {baseline_part_4_metrics['precision']:.4f} no baseline. Diferença: {precision_delta:+.4f}.",
    },
    {
        "aspecto": "balanced_accuracy",
        "conclusao": f"A balanced accuracy do SVM com AG foi {optimized_ga_metrics['balanced_accuracy']:.4f}, contra {baseline_part_4_metrics['balanced_accuracy']:.4f} no baseline. Diferença: {balanced_accuracy_delta:+.4f}.",
    },
    {
        "aspecto": "interpretacao",
        "conclusao": "Como se trata de um problema de saúde, a redução de falsos negativos é o objetivo mais crítico. O ganho de recall indica que o SVM com AG recupera mais casos positivos, mas esse resultado deve ser lido junto com a queda de precision, F1 e balanced accuracy.",
    },
    {
        "aspecto": "decisao_final",
        "conclusao": "O Algoritmo Genético foi bem-sucedido no objetivo prioritário de reduzir falsos negativos quando apresentou recall superior ao baseline da Parte 4. Esse ganho torna o modelo mais sensível para triagem em saúde, embora o aumento esperado de falsos positivos exija uma etapa posterior de confirmação diagnóstica.",
    },
]

optimization_conclusion = pd.DataFrame(conclusion_rows)
optimization_conclusion


,aspecto,conclusao
0,melhor_individuo_ag,"C=0.01, class_weight=balanced, scaler_type=normalizer, max_iter=5000, fitness=0.2764."
1,comparacao_f1,"O F1 do SVM com AG foi 0.0958, contra 0.1225 no baseline da Parte 4. Diferença: -0.0267."
2,comparacao_recall,"O recall do SVM com AG foi 0.6903, contra 0.5251 no baseline. Diferença: +0.1652."
3,comparacao_precision,"A precision do SVM com AG foi 0.0515, contra 0.0693 no baseline. Diferença: -0.0178."
4,balanced_accuracy,"A balanced accuracy do SVM com AG foi 0.5638, contra 0.6066 no baseline. Diferença: -0.0428."
5,interpretacao,"Como se trata de um problema de saúde, a redução de falsos negativos é o objetivo mais crítico. O ganho de recall indica que o SVM com AG recupera mais casos positivos, mas esse resultado deve ser lido junto com a queda de precision, F1 e balanced accuracy."
6,decisao_final,"O Algoritmo Genético foi bem-sucedido no objetivo prioritário de reduzir falsos negativos quando apresentou recall superior ao baseline da Parte 4. Esse ganho torna o modelo mais sensível para triagem em saúde, embora o aumento esperado de falsos positivos exija uma etapa posterior de confirmação diagnóstica."


---

## Resultado da Parte 7

Este notebook implementa um Algoritmo Genético para otimizar hiperparâmetros do SVM linear (`LinearSVC`) usando as bases encoded finais da Parte 2.

A avaliação dos indivíduos ocorre por validação cruzada estratificada com 5 folds, usando uma fitness voltada à classe positiva:

```text
fitness = 0.7 * F1 + 0.3 * recall
```

Foram definidos três experimentos com diferentes configurações de população, mutação e gerações. Ao final, o melhor indivíduo entre os experimentos é usado para treinar um modelo final e avaliar seu desempenho na amostra de teste.

A seção final compara o SVM otimizado por Algoritmo Genético com o baseline SVM da Parte 4, usando a mesma escala de amostra de teste. A interpretação prioriza recall por se tratar de um contexto de saúde, no qual falsos negativos são mais críticos. O resultado deve ser apresentado como um trade-off: maior sensibilidade para capturar positivos, com possível aumento de falsos positivos e queda em métricas globais.

A conclusão operacional deve ser lida a partir da tabela `optimization_conclusion`. Se o recall do SVM com AG ficar acima do baseline da Parte 4, o Algoritmo Genético terá sido bem-sucedido no objetivo prioritário de reduzir falsos negativos. Caso isso venha acompanhado de queda em precision, F1 ou balanced accuracy, o resultado deve ser apresentado como um trade-off esperado em triagem de saúde: maior sensibilidade para capturar positivos, com mais falsos positivos e necessidade de confirmação diagnóstica posterior.

O monitoramento do AG é registrado em dois níveis: `experiment_1_history`, `experiment_2_history` e `experiment_3_history` armazenam a evolução por geração; `ga_experiment_summary` consolida a configuração, o melhor fitness e o tempo total de cada experimento. Esses logs permitem rastrear desempenho, convergência e custo computacional entre as configurações testadas.

